<img src="https://hilpisch.com/tds_logo.png" width="20%" align="right">

# The Data Scientist
## Book 3 · Statistics, Machine Learning, and Model Evaluation
### Notebook 04 · Feature Thinking
© Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.x<br> The Python Quants GmbH | https://tpq.io<br>

https://thedatascientist.dev | https://linktr.ee/dyjh

### Why this notebook exists
Chapter 6 treats features as design choices. The same rows can look very different to
a model depending on how the inputs are represented.

## Setup
We make the notebook portable first so the helper module and data paths stay
stable across environments.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

BOOK_NAME = "3_data"
REPO_URL = "https://github.com/yhilpisch/tdscode"

cwd = Path.cwd().resolve()
BOOK_ROOT = None
for candidate in (cwd, *cwd.parents):
    if candidate.name == BOOK_NAME and (candidate / "notebooks").exists():
        BOOK_ROOT = candidate
        break
    if (candidate / BOOK_NAME / "notebooks").exists():
        BOOK_ROOT = candidate / BOOK_NAME
        break

if BOOK_ROOT is None:
    repo_root = Path("/content/tdscode")
    if not repo_root.exists():
        # Clone the companion repo once in Colab.
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(repo_root)],
            check=True,
        )
    BOOK_ROOT = repo_root / BOOK_NAME

os.chdir(BOOK_ROOT)

# Make the book root and the helper folder importable.
for path in (BOOK_ROOT, BOOK_ROOT / "code"):
    if path.exists() and str(path) not in sys.path:
        sys.path.insert(0, str(path))

requirements = BOOK_ROOT / "requirements.txt"
if requirements.exists() and "google.colab" in sys.modules:
    # Keep Colab aligned with the book dependencies.
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)],
        check=True,
    )

print("Book root:", BOOK_ROOT)


## Load the Churn Helper
The helper module holds the shared feature and preprocessing definitions. That
lets the notebook focus on feature thinking instead of setup noise.


In [ ]:
from pathlib import Path
import importlib.util

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

module_path = BOOK_ROOT / 'code' / 'churn_report.py'
spec = importlib.util.spec_from_file_location('churn_report', module_path)
assert spec is not None and spec.loader is not None
churn_report = importlib.util.module_from_spec(spec)
spec.loader.exec_module(churn_report)

print(f'Loaded {module_path.resolve()}')


## Inspect the Raw Customer Table
Before we encode anything, we inspect the customer table and a few churn rates
by group. That keeps feature design tied to patterns in the original data.


In [ ]:
df = churn_report.load_customers()

print(df.head().to_string(index=False))
print()
print(df.dtypes.to_string())
print()
print('churn rate by plan type:')
print(df.groupby('plan_type')['churned'].mean().sort_values(ascending=False))
print()
print('churn rate by country:')
print(df.groupby('country')['churned'].mean().sort_values(ascending=False))


## Build the Preprocessing and Model Pipeline
The next cell separates features from the target, applies scaling and one-hot
encoding, and fits a baseline logistic regression pipeline.


In [ ]:
feature_frame = df[churn_report.FEATURE_COLUMNS]
target = df[churn_report.TARGET_COLUMN]

# Split off held-out data before preprocessing.
X_train, X_test, y_train, y_test = train_test_split(
    feature_frame,
    target,
    test_size=0.25,
    random_state=42,
    stratify=target,
)

# Scale numeric features and one-hot encode categoricals.
preprocessor = ColumnTransformer(
    [
        ('num', StandardScaler(), churn_report.NUMERIC_COLUMNS),
        (
            'cat',
            OneHotEncoder(handle_unknown='ignore', sparse_output=False),
            churn_report.CATEGORICAL_COLUMNS,
        ),
    ]
)
pipe = Pipeline(
    [
        ('pre', preprocessor),
        ('model', LogisticRegression(max_iter=1000)),
    ]
)
pipe.fit(X_train, y_train)

feature_names = pipe.named_steps['pre'].get_feature_names_out()
coef_frame = pd.Series(pipe.named_steps['model'].coef_.ravel(), index=feature_names)
coef_frame = coef_frame.sort_values(key=lambda s: s.abs(), ascending=False)

print('transformed shape =', pipe.named_steps['pre'].transform(X_train).shape)
print()
print(coef_frame.head(12).to_string())


## Inspect One Transformed Example
The last step looks at one transformed row so the delegate can see that feature
engineering changes the representation, not just the model object.


In [ ]:
transformed_row = pipe.named_steps['pre'].transform(X_test.iloc[[0]])
print(transformed_row)


### Where We Are Heading Next
Chapter 7 steps back from individual features and introduces clustering and
dimensionality reduction as exploration tools.

<img src="https://hilpisch.com/tds_logo.png" width="20%" align="right">